# Gated Hybrid SNR-aware Attention

This notebook evaluates the automatic hybrid system:

```text
IQ signal -> lightweight SNR gate -> choose attention model -> modulation prediction
```

Comparison in this notebook:

- **Manual hybrid:** uses true SNR label: `SNR <= 0 dB -> differential attention`, `SNR > 0 dB -> normal attention`
- **Gated hybrid:** uses the trained SNR-gate prediction instead of true SNR

This tests whether the hybrid approach can be automated without manually knowing SNR.

In [ ]:
# CELL 1: Setup repo and paths
import os
import sys
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

import pandas as pd
from IPython.display import Image, display, FileLink

os.environ['KERAS_BACKEND'] = 'tensorflow'

REPO_URL = 'https://github.com/akshlabh/amr-5-class.git'
WORK_DIR = Path('/kaggle/working/amr-5-class')

# Edit this manually if your Kaggle dataset path is different.
DATASET_CANDIDATES = [
    Path('/kaggle/input/datasets/gustavopolicarpo/rml201610a-dict/RML2016.10a_dict.dat'),
    Path('/kaggle/input/rml201610a-dict/RML2016.10a_dict.dat'),
    Path('/kaggle/input/radioml2016-10a/RML2016.10a_dict.pkl'),
    Path('/kaggle/input/radioml2016-10a/RML2016.10a_dict.dat'),
    Path('data/RML2016.10a_5class.pkl'),
    Path('data/RML2016.10a_dict.pkl'),
    Path('data/RML2016.10a_dict.dat'),
]

def find_attached_repo():
    input_root = Path('/kaggle/input')
    if not input_root.exists():
        return None
    for root in input_root.glob('**'):
        if (root / 'src' / 'train.py').exists() and (root / 'configs').exists():
            return root
    return None

if (Path.cwd() / 'src' / 'train.py').exists():
    WORK_DIR = Path.cwd()
    print('Using current repo:', WORK_DIR)
elif (WORK_DIR / 'src' / 'train.py').exists():
    print('Using existing repo:', WORK_DIR)
else:
    attached = find_attached_repo()
    if attached is not None:
        print('Copying attached repo from:', attached)
        if WORK_DIR.exists():
            shutil.rmtree(WORK_DIR)
        shutil.copytree(attached, WORK_DIR)
    else:
        print('Cloning repo from GitHub...')
        subprocess.run(['git', 'clone', REPO_URL, str(WORK_DIR)], check=True)

os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

DATASET = next((p for p in DATASET_CANDIDATES if p.exists()), None)
assert DATASET is not None, 'Dataset not found. Set DATASET manually in this cell.'

NORMAL_DIR = Path('experiments/5class_attention')
DIFF_DIR = Path('experiments/5class_diffattention')
GATE_DIR = Path('experiments/snr_gate_lightweight')
GATED_DIR = Path('experiments/5class_gated_hybrid_snr_aware_attention')

print('Working dir:', Path.cwd())
print('Dataset    :', DATASET)
print('Dataset OK :', DATASET.exists())

In [ ]:
# CELL 2: Check required files and checkpoints
required_files = [
    'src/evaluate_gated_hybrid_snr_aware_attention.py',
    'src/models/mcldnn_attention.py',
    'src/models/mcldnn_diffattention.py',
    'src/models/snr_gate.py',
    'src/features/signal_features.py',
    'src/train.py',
    'src/train_snr_gate.py',
    'configs/exp_5class_attention.yaml',
    'configs/exp_5class_diffattention.yaml',
    'configs/exp_snr_gate.yaml',
]

for f in required_files:
    print(('OK      ' if Path(f).exists() else 'MISSING ') + f)
    assert Path(f).exists(), f'Missing required file: {f}'

normal_weights = NORMAL_DIR / 'checkpoints/best_model.weights.h5'
diff_weights = DIFF_DIR / 'checkpoints/best_model.weights.h5'
gate_weights = GATE_DIR / 'checkpoints/best_model.weights.h5'
gate_scaler = GATE_DIR / 'results/snr_gate_feature_scaler.npz'
gate_metadata = GATE_DIR / 'results/snr_gate_metadata.json'

print('\nCheckpoint status:')
for p in [normal_weights, diff_weights, gate_weights, gate_scaler, gate_metadata]:
    print(('OK      ' if p.exists() else 'MISSING ') + str(p))

In [ ]:
# CELL 3: Train any missing source models only if needed
# If you already placed results/weights in the repo, this cell will skip training.

jobs = [
    ('normal_attention', normal_weights, ['src/train.py', '--config', 'configs/exp_5class_attention.yaml']),
    ('diff_attention', diff_weights, ['src/train.py', '--config', 'configs/exp_5class_diffattention.yaml']),
    ('snr_gate', gate_weights, ['src/train_snr_gate.py', '--config', 'configs/exp_snr_gate.yaml']),
]

for name, weights, base_cmd in jobs:
    if weights.exists():
        print(f'{name}: checkpoint found, skipping training.')
    else:
        print(f'{name}: checkpoint missing, training now...')
        cmd = [sys.executable, *base_cmd, '--datasetpath', str(DATASET)]
        print('Running:', ' '.join(cmd))
        subprocess.run(cmd, check=True)
        assert weights.exists(), f'Training finished but weights missing: {weights}'

assert normal_weights.exists()
assert diff_weights.exists()
assert gate_weights.exists()
assert gate_scaler.exists(), 'SNR gate scaler missing. Re-run SNR gate training notebook/script.'

In [ ]:
# CELL 4: Evaluate gated hybrid vs manual hybrid
if GATED_DIR.exists():
    print('Removing old gated-hybrid outputs:', GATED_DIR)
    shutil.rmtree(GATED_DIR)

cmd = [
    sys.executable, '-u', 'src/evaluate_gated_hybrid_snr_aware_attention.py',
    '--datasetpath', str(DATASET),
    '--normal-weights', str(normal_weights),
    '--diff-weights', str(diff_weights),
    '--gate-weights', str(gate_weights),
    '--gate-scaler', str(gate_scaler),
    '--gate-metadata', str(gate_metadata),
    '--output-dir', str(GATED_DIR),
    '--threshold-db', '0',
]
print('Running:', ' '.join(cmd))

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, process.args)

print('Gated-hybrid evaluation finished.')
print('Results:', (GATED_DIR / 'results/test_score.csv').exists())

In [ ]:
# CELL 5: Display comparison tables
score = pd.read_csv(GATED_DIR / 'results/test_score.csv')
per_snr = pd.read_csv(GATED_DIR / 'results/manual_vs_gated_acc_per_snr.csv')
routing = pd.read_csv(GATED_DIR / 'results/routing_summary.csv')
route_conf = pd.read_csv(GATED_DIR / 'results/snr_gate_route_confusion_normalized.csv')

print('Overall model comparison')
display(score)

print('SNR-gate routing summary')
display(routing)

print('Manual hybrid vs gated hybrid by SNR')
display(per_snr)

print('SNR-gate route confusion')
display(route_conf)

In [ ]:
# CELL 6: Display important plots
figs = [
    GATED_DIR / 'figures/manual_vs_gated_hybrid_acc_vs_snr.png',
    GATED_DIR / 'figures/gated_minus_manual_delta_by_snr.png',
    GATED_DIR / 'figures/snr_gate_route_confusion.png',
    GATED_DIR / 'figures/confusion_manual_hybrid_all_snrs.png',
    GATED_DIR / 'figures/confusion_gated_hybrid_all_snrs.png',
]

for fig in figs:
    print(fig)
    display(Image(filename=str(fig)))

In [ ]:
# CELL 7: Create repo-ready zip for only gated-hybrid results
stamp = datetime.now().strftime('%Y%m%d_%H%M')
zip_base = Path('/kaggle/working') / f'gated_hybrid_snr_aware_attention_repo_ready_{stamp}'
zip_path = shutil.make_archive(
    str(zip_base),
    'zip',
    root_dir=str(WORK_DIR),
    base_dir='experiments/5class_gated_hybrid_snr_aware_attention',
)

print('Created repo-ready zip:', zip_path)
print('\nExtract this zip at repo root. It will create/update:')
print('  experiments/5class_gated_hybrid_snr_aware_attention/')
print('\nIncluded files:')
for path in sorted(GATED_DIR.rglob('*')):
    if path.is_file():
        print(' -', Path('experiments/5class_gated_hybrid_snr_aware_attention') / path.relative_to(GATED_DIR))

display(FileLink(zip_path))